# Bundles Advanced

Uses a multi-trace bundle for diff, metric, relationship, and multi_trace surface checks.

Every `COVERAGE_CALLS` entry below is resolved against the current TorchLens API in this notebook.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
from torch import nn
from _shared import audit_touch_items, summarize_value  # noqa: E402


class AuditBufferModel(nn.Module):
    """Tiny model with a registered buffer for BufferLog coverage."""

    def __init__(self) -> None:
        """Initialize one linear layer and one buffer."""

        super().__init__()
        self.linear = nn.Linear(4, 2)
        self.register_buffer("offset", torch.ones(1, 4))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the buffer before the linear layer."""

        return self.linear(x + self.offset)


model = tiny_model(seed=0).train()
x = torch.randn(1, 4, requires_grad=True)
log = tl.trace(
    model,
    x,
    layers_to_save="all",
    save_grads=True,
    save_arg_values=True,
    save_rng_states=True,
    train_mode=True,
    vis_opt="none",
)
loss = log.layer_list[-1].out.sum()
log.log_backward(loss)
layer_pass = log.layer_list[1]
layer_log = log.layer_logs[layer_pass.layer_label_no_pass]
module_log = next(iter(log.modules))
module_pass = next(iter(module_log.ops.values()))
param_log = next(iter(log.params))
grad_fn_log = next(iter(log.grad_fns))
grad_fn_pass = next(iter(grad_fn_log.ops.values()))

buffer_log_model = AuditBufferModel().eval()
buffer_log_capture = tl.trace(
    buffer_log_model, torch.randn(1, 4), layers_to_save="all", vis_opt="none"
)
buffer_log = next(iter(buffer_log_capture.buffers))

clean, corrupt = make_clean_corrupt_pair(seed=0)
bundle = tl.bundle(
    [
        ("baseline", log),
        ("corrupt", tl.trace(tiny_model(seed=1).eval(), corrupt, vis_opt="none")),
    ],
    baseline="baseline",
)
AUDIT_CONTEXT = {
    "tl": tl,
    "Trace": log,
    "LayerLog": layer_log,
    "OpLog": layer_pass,
    "ModuleLog": module_log,
    "ModuleCallLog": module_pass,
    "ParamLog": param_log,
    "BufferLog": buffer_log,
    "GradFnLog": grad_fn_log,
    "GradFnCallLog": grad_fn_pass,
    "Bundle": bundle,
}
print(
    f"audit context: {len(log.layer_list)} layer ops, {len(log.modules)} modules, {len(log.grad_fns)} grad fns"
)

## Bundles Advanced: concrete workflow

This cell demonstrates the user-facing workflow for this notebook before the manifest sweep.

In [ ]:
COVERAGE_CALLS = [
    "tl.multi_trace.METRIC_REGISTRY",
    "tl.multi_trace.SuperOp",
    "tl.multi_trace.Supergraph",
    "tl.multi_trace.SupergraphNode",
    "tl.multi_trace.TopologyDiff",
    "tl.multi_trace.build_supergraph",
    "tl.multi_trace.compare_topology",
    "tl.multi_trace.cosine_distance",
    "tl.multi_trace.pearson_correlation_distance",
    "tl.multi_trace.relative_l1_scalar",
    "tl.multi_trace.relative_l2",
    "tl.multi_trace.resolve_metric",
]


class BundleDemoModel(nn.Module):
    """Small same-topology model for bundle comparisons."""

    def __init__(self, seed: int) -> None:
        """Initialize deterministic demo weights."""

        super().__init__()
        torch.manual_seed(seed)
        self.linear = nn.Linear(4, 4)
        self.relu = nn.ReLU()
        self.readout = nn.Linear(4, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run one hidden layer and a readout."""

        return self.readout(self.relu(self.linear(x)))


bundle_x = torch.randn(1, 4)
bundle_members = {
    "base": tl.trace(BundleDemoModel(0).eval(), bundle_x, layers_to_save="all", vis_opt="none"),
    "alt_weights": tl.trace(
        BundleDemoModel(1).eval(), bundle_x, layers_to_save="all", vis_opt="none"
    ),
    "alt_input": tl.trace(
        BundleDemoModel(0).eval(), bundle_x + 0.25, layers_to_save="all", vis_opt="none"
    ),
}
advanced = tl.bundle(bundle_members, baseline="base")
layer_label = bundle_members["base"].layer_list[1].layer_label_no_pass
super_layer = advanced.layers[layer_label]
comparison = advanced.compare_at(layer_label)
delta = advanced.delta_map("relative_l2")
changed = advanced.most_changed(top_k=3)
print("members", list(advanced.names))
print("super layer", type(super_layer).__name__, list(super_layer.members))
print("compare_at shape", tuple(comparison.shape))
print("delta sample", list(delta.items())[:3])
print("most changed", changed)
print("multi_trace", [name for name in dir(tl.multi_trace) if not name.startswith("_")][:8])

## Bundles Advanced coverage cluster 1

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.multi_trace.METRIC_REGISTRY",
    "tl.multi_trace.SuperOp",
    "tl.multi_trace.Supergraph",
    "tl.multi_trace.SupergraphNode",
    "tl.multi_trace.TopologyDiff",
    "tl.multi_trace.build_supergraph",
    "tl.multi_trace.compare_topology",
    "tl.multi_trace.cosine_distance",
    "tl.multi_trace.pearson_correlation_distance",
    "tl.multi_trace.relative_l1_scalar",
    "tl.multi_trace.relative_l2",
    "tl.multi_trace.resolve_metric",
]

audit_touch_items("Bundles Advanced coverage cluster 1", ITEMS, AUDIT_CONTEXT)

In [ ]:
COVERAGE_CALLS = [
    "tl.multi_trace.METRIC_REGISTRY",
    "tl.multi_trace.SuperOp",
    "tl.multi_trace.Supergraph",
    "tl.multi_trace.SupergraphNode",
    "tl.multi_trace.TopologyDiff",
    "tl.multi_trace.build_supergraph",
    "tl.multi_trace.compare_topology",
    "tl.multi_trace.cosine_distance",
    "tl.multi_trace.pearson_correlation_distance",
    "tl.multi_trace.relative_l1_scalar",
    "tl.multi_trace.relative_l2",
    "tl.multi_trace.resolve_metric",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")